In [ ]:
    # 아래 세 경로를 본인 환경에 맞게 수정하세요.
    intro_path = r"원본 자개소개서의 경로"
    interview_path = r"AI로 생성된 질문 경로"
    output_dir = r"개별 리포트를 저장할 내부경로 설정"

In [3]:
# -*- coding: utf-8 -*-
"""
지원자별 면접 리포트 HTML 생성 (v2)

입력:
  - 자기소개서 원본 엑셀 (시트: "자기소개서" - 면접번호/이름/모집분야/전형단계/질문/답변)
  - 면접질문 생성 결과 엑셀 (시트: "면접질문"
        - 신규 형식: 면접번호/이름/모집분야/주질문/근거구절/질문이유/보조질문
        - 구버전 형식(근거구절 없음): 면접번호/이름/모집분야/주질문/질문이유/보조질문
      둘 다 자동 인식하여 처리)

출력:
  1) 지원자별 개별 HTML 리포트 (이름은 블라인드 처리, 면접번호 기준 파일명)
     - 리포트 상단: 맞춤형 면접질문(주질문 4 + 근거구절 + 질문이유 + 보조질문 3개씩)
     - 리포트 하단: 자기소개서 원본 문항/답변
     - 주질문의 근거구절이 자소서의 몇 번 문항 답변에서 인용되었는지 "출처" 표기
  2) 통합 리포트 (전체리포트.html)
     - 좌측 사이드바에서 면접번호를 클릭하면 우측에 해당 리포트가 표시됨
     - 별도 파일 이동 없이 한 페이지에서 전체 지원자 탐색 가능

사용법:
  main() 함수 안의 경로 3개(intro_path, interview_path, output_dir)를
  본인 환경에 맞게 수정한 뒤 실행하세요.
"""

import os
import re
import html
import base64
import openpyxl


# ----------------------------------------------------------------------
# 로고 설정
# ----------------------------------------------------------------------
# 리포트 상단에 표시할 한국마사회 로고 이미지 파일명.
# main() 실행 시 output_dir(결과 폴더) 안에 이 이름의 파일이 있으면
# 자동으로 base64 인코딩되어 HTML에 내장됩니다
# (인터넷 연결 없이도, PDF 변환 시에도 정상 표시됨).
# 파일이 없으면 로고 없이 정상적으로 리포트가 생성됩니다.
LOGO_FILENAME = "kra_logo.png"

# 실제 로고 경로. main()에서 output_dir을 기준으로 자동 설정되며,
# 직접 다른 경로를 쓰고 싶다면 이 값을 원하는 절대경로로 바꿔주세요.
LOGO_PATH = LOGO_FILENAME

_LOGO_DATA_URI_CACHE = None


def get_logo_data_uri(logo_path=None):
    """로고 이미지를 base64 data URI로 변환. 파일이 없으면 None 반환."""
    global _LOGO_DATA_URI_CACHE

    if logo_path is None:
        logo_path = LOGO_PATH

    if _LOGO_DATA_URI_CACHE is not None:
        return _LOGO_DATA_URI_CACHE or None

    if not os.path.exists(logo_path):
        _LOGO_DATA_URI_CACHE = ""
        return None

    ext = os.path.splitext(logo_path)[1].lower()
    mime_map = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".svg": "image/svg+xml",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }
    mime = mime_map.get(ext, "image/png")

    with open(logo_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("ascii")

    _LOGO_DATA_URI_CACHE = f"data:{mime};base64,{encoded}"
    return _LOGO_DATA_URI_CACHE


# ----------------------------------------------------------------------
# 데이터 로드
# ----------------------------------------------------------------------
def load_intro(path):
    """자기소개서 원본: 면접번호 -> {모집분야, 전형단계, qa:[{question, answer}]}"""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["자기소개서"]

    data = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[0] is None:
            continue
        exam_no, name, field, status, question, answer = row[:6]
        if exam_no not in data:
            data[exam_no] = {
                "모집분야": field,
                "전형단계": status,
                "qa": [],
            }
        data[exam_no]["qa"].append({
            "question": question or "",
            "answer": answer or "",
        })
    return data


def load_interview_questions(path):
    """면접질문 결과: 면접번호 -> {모집분야, main_questions:[{question, source_quote, reason, sub_questions:[...]}]}

    - 헤더를 읽어서 "근거구절" 컬럼 존재 여부를 자동 판단 (구버전 파일과 호환)
    """
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["면접질문"]

    rows = list(ws.iter_rows(min_row=1, values_only=True))
    if not rows:
        return {}

    header = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx = {name: i for i, name in enumerate(header)}

    def get(row, key, default=None):
        idx = col_idx.get(key)
        if idx is None or idx >= len(row):
            return default
        return row[idx]

    data = {}
    for row in rows[1:]:
        if row[0] is None:
            continue

        exam_no = get(row, "면접번호")
        field = get(row, "모집분야")
        question = get(row, "주질문", "")
        source_quote = get(row, "근거구절", "")
        reason = get(row, "질문이유", "")
        sub_text = get(row, "보조질문", "")

        if exam_no not in data:
            data[exam_no] = {"모집분야": field, "main_questions": []}

        sub_list = []
        if sub_text:
            sub_list = [s.strip() for s in str(sub_text).split("\n") if s.strip()]

        data[exam_no]["main_questions"].append({
            "question": question or "",
            "source_quote": source_quote or "",
            "reason": reason or "",
            "sub_questions": sub_list,
        })
    return data


# ----------------------------------------------------------------------
# 근거구절 -> 자소서 문항 매칭
# ----------------------------------------------------------------------
def _normalize(text):
    return re.sub(r"\s+", "", str(text or ""))


def find_source_question_no(source_quote, qa_list, min_len=8):
    """source_quote가 어느 자소서 문항(answer)에서 인용되었는지 추정.
    공백을 제거한 뒤 부분 문자열 포함 여부로 판단.
    너무 짧은 인용(min_len 미만)은 매칭하지 않음(과도한 오탐 방지)."""
    quote_norm = _normalize(source_quote)
    if len(quote_norm) < min_len:
        return None

    for i, qa in enumerate(qa_list, start=1):
        answer_norm = _normalize(qa.get("answer", ""))
        if quote_norm and quote_norm in answer_norm:
            return i

        # 인용문이 약간 다를 경우를 대비해, 앞부분 일부만으로도 매칭 시도
        head = quote_norm[:min(len(quote_norm), 20)]
        if len(head) >= min_len and head in answer_norm:
            return i

    return None


# ----------------------------------------------------------------------
# 공통 스타일
# ----------------------------------------------------------------------
COMMON_STYLE = """
  * { box-sizing: border-box; }
  body {
    font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif;
    color: #222;
    line-height: 1.6;
  }
  .container {
    max-width: 980px;
    margin: 0 auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .header {
    background: #144782;
    color: #fff;
    padding: 28px 32px;
  }
  .header-top {
    display: flex;
    align-items: center;
    gap: 14px;
    margin-bottom: 6px;
  }
  .logo-img {
    height: 36px;
    width: auto;
    flex-shrink: 0;
    background: #fff;
    border-radius: 4px;
    padding: 4px 8px;
  }
  .header h1 {
    margin: 0;
    font-size: 22px;
  }
  .header .meta {
    font-size: 14px;
    color: #cfe0f5;
  }
  .badge {
    display: inline-block;
    background: rgba(255,255,255,0.18);
    border-radius: 4px;
    padding: 2px 8px;
    font-size: 12px;
    margin-right: 6px;
  }
  .section {
    padding: 24px 32px;
    border-bottom: 1px solid #eee;
  }
  .section h2 {
    font-size: 18px;
    color: #144782;
    border-left: 5px solid #00A597;
    padding-left: 10px;
    margin-top: 0;
  }
  /* 자기소개서 */
  .intro-item {
    background: #fafbfc;
    border: 1px solid #e3e7eb;
    border-radius: 8px;
    padding: 16px 18px;
    margin-bottom: 14px;
  }
  .intro-item .intro-no {
    display: inline-block;
    background: #607d8b;
    color: #fff;
    font-size: 11px;
    font-weight: bold;
    border-radius: 4px;
    padding: 2px 6px;
    margin-right: 6px;
  }
  .intro-q {
    font-weight: bold;
    color: #144782;
    margin-bottom: 8px;
  }
  .intro-a {
    white-space: pre-wrap;
    color: #444;
    font-size: 14px;
  }
  /* 면접질문 */
  .main-q-block {
    border: 1px solid #d9e2ec;
    border-radius: 10px;
    margin-bottom: 18px;
    overflow: hidden;
  }
  .main-q-header {
    background: #eaf1fb;
    padding: 14px 18px;
  }
  .main-q-no {
    display: inline-block;
    background: #144782;
    color: #fff;
    font-size: 12px;
    font-weight: bold;
    border-radius: 4px;
    padding: 2px 8px;
    margin-right: 8px;
  }
  .main-q-text {
    font-weight: bold;
    font-size: 15px;
    color: #1a1a1a;
  }
  .source-box {
    margin-top: 10px;
    font-size: 13px;
    color: #5d4037;
    background: #fff3e0;
    border-left: 3px solid #ff9800;
    border-radius: 6px;
    padding: 8px 12px;
  }
  .source-box .label {
    font-weight: bold;
    margin-right: 4px;
    color: #e65100;
  }
  .source-box .quote {
    font-style: italic;
  }
  .source-box .source-ref {
    display: block;
    margin-top: 4px;
    font-size: 12px;
    color: #9e7b52;
  }
  .reason-box {
    margin-top: 10px;
    font-size: 13px;
    color: #00695c;
    background: #e6f4f1;
    border-radius: 6px;
    padding: 8px 12px;
  }
  .reason-box .label {
    font-weight: bold;
    margin-right: 4px;
  }
  .sub-q-list {
    padding: 14px 18px 16px 18px;
  }
  .sub-q-list .sub-title {
    font-size: 13px;
    font-weight: bold;
    color: #888;
    margin-bottom: 8px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
  }
  .sub-q-item {
    padding: 8px 0 8px 28px;
    position: relative;
    font-size: 14px;
    color: #333;
    border-top: 1px dashed #e0e0e0;
  }
  .sub-q-item:first-child { border-top: none; }
  .sub-q-item::before {
    content: "▸";
    position: absolute;
    left: 6px;
    color: #00A597;
    font-weight: bold;
  }
  .footer {
    padding: 16px 32px;
    font-size: 12px;
    color: #999;
    text-align: center;
  }
"""


# ----------------------------------------------------------------------
# 렌더링 함수
# ----------------------------------------------------------------------
def render_intro_html(qa_list):
    parts = []
    for i, qa in enumerate(qa_list, start=1):
        q = html.escape(str(qa["question"]))
        a = html.escape(str(qa["answer"]))
        parts.append(f"""
        <div class="intro-item">
          <div class="intro-q"><span class="intro-no">문항 {i}</span>{q}</div>
          <div class="intro-a">{a}</div>
        </div>""")
    return "\n".join(parts) if parts else "<p>자기소개서 데이터가 없습니다.</p>"


# ----------------------------------------------------------------------
# 평가요소 라벨
# ----------------------------------------------------------------------
# 주질문 순서대로 매핑되는 평가요소명. 5개를 기본으로 하며,
# main_questions 개수가 이보다 적거나 많으면 부족/초과분은 "주질문 N"으로 표시한다.
EVAL_FACTOR_LABELS = ["정신자세", "전문지식", "발표력", "품행", "발전가능성"]


def get_main_q_label(index_1based):
    """주질문 순번(1-based)에 해당하는 평가요소명 라벨을 반환"""
    if 1 <= index_1based <= len(EVAL_FACTOR_LABELS):
        return EVAL_FACTOR_LABELS[index_1based - 1]
    return f"주질문 {index_1based}"


def render_questions_html(main_questions, qa_list):
    parts = []
    for i, mq in enumerate(main_questions, start=1):
        q = html.escape(str(mq["question"]))
        reason = html.escape(str(mq.get("reason", "")))
        source_quote = str(mq.get("source_quote", "")).strip()

        sub_items = "".join(
            f'<div class="sub-q-item">{html.escape(str(s))}</div>'
            for s in mq.get("sub_questions", [])
        )

        reason_html = (
            f'<div class="reason-box"><span class="label">질문이유</span>{reason}</div>'
            if reason else ""
        )

        source_html = ""
        if source_quote:
            quote_escaped = html.escape(source_quote)
            q_no = find_source_question_no(source_quote, qa_list)
            if q_no:
                ref_text = f"출처: 자기소개서 문항 {q_no}"
            else:
                ref_text = "출처: 자기소개서 (정확한 문항 매칭 안됨)"
            source_html = (
                f'<div class="source-box">'
                f'<span class="label">근거구절</span>'
                f'<span class="quote">“{quote_escaped}”</span>'
                f'<span class="source-ref">{ref_text}</span>'
                f'</div>'
            )

        parts.append(f"""
        <div class="main-q-block">
          <div class="main-q-header">
            <div><span class="main-q-no">{html.escape(get_main_q_label(i))}</span></div>
            <div class="main-q-text">{q}</div>
            {source_html}
            {reason_html}
          </div>
          <div class="sub-q-list">
            <div class="sub-title">보조 질문</div>
            {sub_items if sub_items else '<div class="sub-q-item">(보조질문 없음)</div>'}
          </div>
        </div>""")
    return "\n".join(parts) if parts else "<p>생성된 면접 질문이 없습니다.</p>"


def render_report_body(exam_no, field, intro_qa, main_questions):
    """리포트 본문(헤더+섹션들)만 렌더링 - 개별 파일과 통합 페이지에서 공통 사용"""

    logo_uri = get_logo_data_uri()
    logo_html = (
        f'<img src="{logo_uri}" alt="한국마사회 로고" class="logo-img">'
        if logo_uri else ""
    )

    return f"""
  <div class="header">
    <div class="header-top">
      {logo_html}
      <h1>2026년 한국마사회 전임직·위촉직채용 개별면접질문</h1>
    </div>
    <div class="meta">
      <span class="badge">면접번호: {html.escape(str(exam_no))}</span>
      <span class="badge">모집분야: {html.escape(str(field or ""))}</span>
      <span class="badge">이름: ●●●(블라인드)</span>
    </div>
  </div>

  <div class="section">
    <h2>1. 맞춤형 면접 질문 (정신자세 · 전문지식 · 발표력 · 품행 · 발전가능성 - 각 보조질문 3개, 총 15개)</h2>
    {render_questions_html(main_questions, intro_qa)}
  </div>

  <div class="section">
    <h2>2. 자기소개서 원본</h2>
    {render_intro_html(intro_qa)}
  </div>

  <div class="footer">
    본 리포트는 생성형 AI 기술을 활용하여 지원자가 작성한 자기소개서 내용을 바탕으로 맞춤 제작되었습니다.
  </div>
"""


# ----------------------------------------------------------------------
# 개별 리포트 HTML
# ----------------------------------------------------------------------
INDIVIDUAL_TEMPLATE = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>2026년 한국마사회 위촉직채용 개별면접질문 - {exam_no}</title>
<style>
  body {{ background: #f4f6f8; margin: 0; padding: 24px; }}
{common_style}
</style>
</head>
<body>
<div class="container">
{body}
</div>
</body>
</html>
"""


def build_individual_report(exam_no, field, intro_qa, main_questions):
    body = render_report_body(exam_no, field, intro_qa, main_questions)
    return INDIVIDUAL_TEMPLATE.format(
        exam_no=html.escape(str(exam_no)),
        common_style=COMMON_STYLE,
        body=body,
    )


# ----------------------------------------------------------------------
# 통합 리포트(사이드바) HTML
# ----------------------------------------------------------------------
COMBINED_TEMPLATE = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>2026년 한국마사회 위촉직채용 개별면접질문 - 통합 보기</title>
<style>
  * {{ box-sizing: border-box; }}
  html, body {{
    margin: 0;
    height: 100%;
    font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif;
    background: #f4f6f8;
  }}
  .layout {{
    display: flex;
    height: 100vh;
  }}
  .sidebar {{
    width: 260px;
    flex-shrink: 0;
    background: #0f2f57;
    color: #fff;
    overflow-y: auto;
    padding: 16px 0;
  }}
  .sidebar h2 {{
    font-size: 14px;
    color: #9fc1e8;
    padding: 0 16px;
    margin: 0 0 10px 0;
    text-transform: uppercase;
    letter-spacing: 1px;
  }}
  .sidebar .item {{
    display: block;
    width: 100%;
    text-align: left;
    background: none;
    border: none;
    color: #d6e4f7;
    padding: 10px 16px;
    font-size: 14px;
    cursor: pointer;
    border-left: 3px solid transparent;
  }}
  .sidebar .item .field {{
    display: block;
    font-size: 11px;
    color: #8fb2dd;
    margin-top: 2px;
  }}
  .sidebar .item:hover {{
    background: #16407a;
  }}
  .sidebar .item.active {{
    background: #144782;
    border-left-color: #00A597;
    color: #fff;
    font-weight: bold;
  }}
  .content {{
    flex: 1;
    overflow-y: auto;
    padding: 24px;
  }}
  .report-panel {{
    display: none;
    max-width: 980px;
    margin: 0 auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    overflow: hidden;
  }}
  .report-panel.active {{
    display: block;
  }}
  .placeholder {{
    max-width: 980px;
    margin: 60px auto;
    text-align: center;
    color: #999;
    font-size: 15px;
  }}
{common_style}
</style>
</head>
<body>
<div class="layout">
  <div class="sidebar">
    <h2>지원자 목록 ({count}명)</h2>
    {sidebar_items}
  </div>
  <div class="content">
    <div class="placeholder" id="placeholder">좌측 목록에서 면접번호를 선택하세요.</div>
    {panels}
  </div>
</div>
<script>
  function showReport(examNo) {{
    document.querySelectorAll('.report-panel').forEach(function(p) {{
      p.classList.toggle('active', p.id === 'panel-' + examNo);
    }});
    document.querySelectorAll('.sidebar .item').forEach(function(b) {{
      b.classList.toggle('active', b.dataset.examNo === examNo);
    }});
    var ph = document.getElementById('placeholder');
    if (ph) ph.style.display = 'none';
  }}

  // 첫 번째 지원자를 기본으로 표시
  document.addEventListener('DOMContentLoaded', function() {{
    var first = document.querySelector('.sidebar .item');
    if (first) showReport(first.dataset.examNo);
  }});
</script>
</body>
</html>
"""


def build_combined_report(all_reports):
    """all_reports: list of (exam_no, field, intro_qa, main_questions)"""
    sidebar_items = []
    panels = []

    for exam_no, field, intro_qa, main_questions in all_reports:
        exam_no_str = html.escape(str(exam_no))
        field_str = html.escape(str(field or ""))

        sidebar_items.append(f"""
        <button class="item" data-exam-no="{exam_no_str}" onclick="showReport('{exam_no_str}')">
          {exam_no_str}
          <span class="field">{field_str}</span>
        </button>""")

        body = render_report_body(exam_no, field, intro_qa, main_questions)
        panels.append(f"""
        <div class="report-panel" id="panel-{exam_no_str}">
        {body}
        </div>""")

    return COMBINED_TEMPLATE.format(
        common_style=COMMON_STYLE,
        count=len(all_reports),
        sidebar_items="\n".join(sidebar_items),
        panels="\n".join(panels),
    )


# ----------------------------------------------------------------------
# main
# ----------------------------------------------------------------------
def main():
    global LOGO_PATH

    # 아래 세 경로를 본인 환경에 맞게 수정하세요.
    intro_path = r"C:\Users\LG\Desktop\마사회 개별 리포트\마사회 자개소개서v2.xlsx"
    interview_path = r"C:\Users\LG\Desktop\마사회 개별 리포트\맞춤화질문 생성결과v3.xlsx"
    output_dir = r"C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)"

    os.makedirs(output_dir, exist_ok=True)

    # output_dir 안에 LOGO_FILENAME(kra_logo.png) 파일이 있으면 자동으로 로고를 삽입합니다.
    # 다른 위치의 로고를 쓰려면 아래 줄에 절대경로를 직접 지정하세요.
    LOGO_PATH = os.path.join(output_dir, LOGO_FILENAME)

    intro_data = load_intro(intro_path)
    interview_data = load_interview_questions(interview_path)

    all_exam_nos = sorted(set(intro_data.keys()) | set(interview_data.keys()), key=str)

    print(f"총 {len(all_exam_nos)}명 처리")

    all_reports = []  # for combined report
    for exam_no in all_exam_nos:
        intro = intro_data.get(exam_no, {"모집분야": "", "qa": []})
        interview = interview_data.get(exam_no, {"모집분야": intro.get("모집분야", ""), "main_questions": []})

        field = intro.get("모집분야") or interview.get("모집분야") or ""
        intro_qa = intro["qa"]
        main_questions = interview["main_questions"]

        # 1) 개별 리포트
        report_html = build_individual_report(exam_no, field, intro_qa, main_questions)
        out_path = os.path.join(output_dir, f"면접리포트_{exam_no}.html")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(report_html)
        print(f"  - {out_path}")

        all_reports.append((exam_no, field, intro_qa, main_questions))

    # 2) 통합 리포트(사이드바)
    combined_html = build_combined_report(all_reports)
    combined_path = os.path.join(output_dir, "전체리포트.html")
    with open(combined_path, "w", encoding="utf-8") as f:
        f.write(combined_html)
    print(f"  - {combined_path}  (좌측 탭 선택 통합 리포트)")

    print("완료")


if __name__ == "__main__":
    main()

총 48명 처리
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-01.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-02.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-03.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-04.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-05.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-06.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-07.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-08.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-09.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-10.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-11.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_B-01.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_B-02.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_C-01.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_C-02.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_C-03.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포

In [ ]:
# -*- coding: utf-8 -*-
"""
지원자별 면접 리포트 HTML 생성 (v2)

입력:
  - 자기소개서 원본 엑셀 (시트: "자기소개서" - 면접번호/이름/모집분야/전형단계/질문/답변)
  - 면접질문 생성 결과 엑셀 (시트: "면접질문"
        - 신규 형식: 면접번호/이름/모집분야/주질문/근거구절/질문이유/보조질문
        - 구버전 형식(근거구절 없음): 면접번호/이름/모집분야/주질문/질문이유/보조질문
      둘 다 자동 인식하여 처리)

출력:
  1) 지원자별 개별 HTML 리포트 (이름은 블라인드 처리, 면접번호 기준 파일명)
     - 리포트 상단: 맞춤형 면접질문(주질문 4 + 근거구절 + 질문이유 + 보조질문 3개씩)
     - 리포트 하단: 자기소개서 원본 문항/답변
     - 주질문의 근거구절이 자소서의 몇 번 문항 답변에서 인용되었는지 "출처" 표기
  2) 통합 리포트 (전체리포트.html)
     - 좌측 사이드바에서 면접번호를 클릭하면 우측에 해당 리포트가 표시됨
     - 별도 파일 이동 없이 한 페이지에서 전체 지원자 탐색 가능

사용법:
  main() 함수 안의 경로 3개(intro_path, interview_path, output_dir)를
  본인 환경에 맞게 수정한 뒤 실행하세요.
"""

import os
import re
import html
import base64
import openpyxl


# ----------------------------------------------------------------------
# 로고 설정
# ----------------------------------------------------------------------
# 리포트 상단에 표시할 한국마사회 로고 이미지 파일명.
# main() 실행 시 output_dir(결과 폴더) 안에 이 이름의 파일이 있으면
# 자동으로 base64 인코딩되어 HTML에 내장됩니다
# (인터넷 연결 없이도, PDF 변환 시에도 정상 표시됨).
# 파일이 없으면 로고 없이 정상적으로 리포트가 생성됩니다.
LOGO_FILENAME = "kra_logo.png"

# 실제 로고 경로. main()에서 output_dir을 기준으로 자동 설정되며,
# 직접 다른 경로를 쓰고 싶다면 이 값을 원하는 절대경로로 바꿔주세요.
LOGO_PATH = LOGO_FILENAME

_LOGO_DATA_URI_CACHE = None


def get_logo_data_uri(logo_path=None):
    """로고 이미지를 base64 data URI로 변환. 파일이 없으면 None 반환."""
    global _LOGO_DATA_URI_CACHE

    if logo_path is None:
        logo_path = LOGO_PATH

    if _LOGO_DATA_URI_CACHE is not None:
        return _LOGO_DATA_URI_CACHE or None

    if not os.path.exists(logo_path):
        _LOGO_DATA_URI_CACHE = ""
        return None

    ext = os.path.splitext(logo_path)[1].lower()
    mime_map = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".svg": "image/svg+xml",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }
    mime = mime_map.get(ext, "image/png")

    with open(logo_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("ascii")

    _LOGO_DATA_URI_CACHE = f"data:{mime};base64,{encoded}"
    return _LOGO_DATA_URI_CACHE


# ----------------------------------------------------------------------
# 데이터 로드
# ----------------------------------------------------------------------
def load_intro(path):
    """자기소개서 원본: 면접번호 -> {모집분야, 전형단계, qa:[{question, answer}]}"""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["자기소개서"]

    data = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[0] is None:
            continue
        exam_no, name, field, status, question, answer = row[:6]
        if exam_no not in data:
            data[exam_no] = {
                "모집분야": field,
                "전형단계": status,
                "qa": [],
            }
        data[exam_no]["qa"].append({
            "question": question or "",
            "answer": answer or "",
        })
    return data


def load_interview_questions(path):
    """면접질문 결과: 면접번호 -> {모집분야, main_questions:[{question, source_quote, reason, sub_questions:[...]}]}

    - 헤더를 읽어서 "근거구절" 컬럼 존재 여부를 자동 판단 (구버전 파일과 호환)
    """
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["면접질문"]

    rows = list(ws.iter_rows(min_row=1, values_only=True))
    if not rows:
        return {}

    header = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx = {name: i for i, name in enumerate(header)}

    def get(row, key, default=None):
        idx = col_idx.get(key)
        if idx is None or idx >= len(row):
            return default
        return row[idx]

    data = {}
    for row in rows[1:]:
        if row[0] is None:
            continue

        exam_no = get(row, "면접번호")
        field = get(row, "모집분야")
        question = get(row, "주질문", "")
        source_quote = get(row, "근거구절", "")
        reason = get(row, "질문이유", "")
        sub_text = get(row, "보조질문", "")

        if exam_no not in data:
            data[exam_no] = {"모집분야": field, "main_questions": []}

        sub_list = []
        if sub_text:
            sub_list = [s.strip() for s in str(sub_text).split("\n") if s.strip()]

        data[exam_no]["main_questions"].append({
            "question": question or "",
            "source_quote": source_quote or "",
            "reason": reason or "",
            "sub_questions": sub_list,
        })
    return data


# ----------------------------------------------------------------------
# 근거구절 -> 자소서 문항 매칭
# ----------------------------------------------------------------------
def _normalize(text):
    return re.sub(r"\s+", "", str(text or ""))


def find_source_question_no(source_quote, qa_list, min_len=8):
    """source_quote가 어느 자소서 문항(answer)에서 인용되었는지 추정.
    공백을 제거한 뒤 부분 문자열 포함 여부로 판단.
    너무 짧은 인용(min_len 미만)은 매칭하지 않음(과도한 오탐 방지)."""
    quote_norm = _normalize(source_quote)
    if len(quote_norm) < min_len:
        return None

    for i, qa in enumerate(qa_list, start=1):
        answer_norm = _normalize(qa.get("answer", ""))
        if quote_norm and quote_norm in answer_norm:
            return i

        # 인용문이 약간 다를 경우를 대비해, 앞부분 일부만으로도 매칭 시도
        head = quote_norm[:min(len(quote_norm), 20)]
        if len(head) >= min_len and head in answer_norm:
            return i

    return None


# ----------------------------------------------------------------------
# 공통 스타일
# ----------------------------------------------------------------------
COMMON_STYLE = """
  * { box-sizing: border-box; }
  body {
    font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif;
    color: #222;
    line-height: 1.6;
  }
  .container {
    max-width: 980px;
    margin: 0 auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .header {
    background: #144782;
    color: #fff;
    padding: 28px 32px;
  }
  .header-top {
    display: flex;
    align-items: center;
    gap: 14px;
    margin-bottom: 6px;
  }
  .logo-img {
    height: 36px;
    width: auto;
    flex-shrink: 0;
    background: #fff;
    border-radius: 4px;
    padding: 4px 8px;
  }
  .header h1 {
    margin: 0;
    font-size: 22px;
  }
  .header .meta {
    font-size: 14px;
    color: #cfe0f5;
  }
  .badge {
    display: inline-block;
    background: rgba(255,255,255,0.18);
    border-radius: 4px;
    padding: 2px 8px;
    font-size: 12px;
    margin-right: 6px;
  }
  .section {
    padding: 24px 32px;
    border-bottom: 1px solid #eee;
  }
  .section h2 {
    font-size: 18px;
    color: #144782;
    border-left: 5px solid #00A597;
    padding-left: 10px;
    margin-top: 0;
  }
  /* 자기소개서 */
  .intro-item {
    background: #fafbfc;
    border: 1px solid #e3e7eb;
    border-radius: 8px;
    padding: 16px 18px;
    margin-bottom: 14px;
  }
  .intro-item .intro-no {
    display: inline-block;
    background: #607d8b;
    color: #fff;
    font-size: 11px;
    font-weight: bold;
    border-radius: 4px;
    padding: 2px 6px;
    margin-right: 6px;
  }
  .intro-q {
    font-weight: bold;
    color: #144782;
    margin-bottom: 8px;
  }
  .intro-a {
    white-space: pre-wrap;
    color: #444;
    font-size: 14px;
  }
  /* 면접질문 */
  .main-q-block {
    border: 1px solid #d9e2ec;
    border-radius: 10px;
    margin-bottom: 18px;
    overflow: hidden;
  }
  .main-q-header {
    background: #eaf1fb;
    padding: 14px 18px;
  }
  .main-q-no {
    display: inline-block;
    background: #144782;
    color: #fff;
    font-size: 12px;
    font-weight: bold;
    border-radius: 4px;
    padding: 2px 8px;
    margin-right: 8px;
  }
  .main-q-text {
    font-weight: bold;
    font-size: 15px;
    color: #1a1a1a;
  }
  .source-box {
    margin-top: 10px;
    font-size: 13px;
    color: #5d4037;
    background: #fff3e0;
    border-left: 3px solid #ff9800;
    border-radius: 6px;
    padding: 8px 12px;
  }
  .source-box .label {
    font-weight: bold;
    margin-right: 4px;
    color: #e65100;
  }
  .source-box .quote {
    font-style: italic;
  }
  .source-box .source-ref {
    display: block;
    margin-top: 4px;
    font-size: 12px;
    color: #9e7b52;
  }
  .reason-box {
    margin-top: 10px;
    font-size: 13px;
    color: #00695c;
    background: #e6f4f1;
    border-radius: 6px;
    padding: 8px 12px;
  }
  .reason-box .label {
    font-weight: bold;
    margin-right: 4px;
  }
  .sub-q-list {
    padding: 14px 18px 16px 18px;
  }
  .sub-q-list .sub-title {
    font-size: 13px;
    font-weight: bold;
    color: #888;
    margin-bottom: 8px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
  }
  .sub-q-item {
    padding: 8px 0 8px 28px;
    position: relative;
    font-size: 14px;
    color: #333;
    border-top: 1px dashed #e0e0e0;
  }
  .sub-q-item:first-child { border-top: none; }
  .sub-q-item::before {
    content: "▸";
    position: absolute;
    left: 6px;
    color: #00A597;
    font-weight: bold;
  }
  .footer {
    padding: 16px 32px;
    font-size: 12px;
    color: #999;
    text-align: center;
  }
"""


# ----------------------------------------------------------------------
# 렌더링 함수
# ----------------------------------------------------------------------
def render_intro_html(qa_list):
    parts = []
    for i, qa in enumerate(qa_list, start=1):
        q = html.escape(str(qa["question"]))
        a = html.escape(str(qa["answer"]))
        parts.append(f"""
        <div class="intro-item">
          <div class="intro-q"><span class="intro-no">문항 {i}</span>{q}</div>
          <div class="intro-a">{a}</div>
        </div>""")
    return "\n".join(parts) if parts else "<p>자기소개서 데이터가 없습니다.</p>"


# ----------------------------------------------------------------------
# 평가요소 라벨
# ----------------------------------------------------------------------
# 주질문 순서대로 매핑되는 평가요소명. 5개를 기본으로 하며,
# main_questions 개수가 이보다 적거나 많으면 부족/초과분은 "주질문 N"으로 표시한다.
EVAL_FACTOR_LABELS = ["정신자세", "전문지식", "발표력", "품행", "발전가능성"]


def get_main_q_label(index_1based):
    """주질문 순번(1-based)에 해당하는 평가요소명 라벨을 반환"""
    if 1 <= index_1based <= len(EVAL_FACTOR_LABELS):
        return EVAL_FACTOR_LABELS[index_1based - 1]
    return f"주질문 {index_1based}"


def render_questions_html(main_questions, qa_list):
    parts = []
    for i, mq in enumerate(main_questions, start=1):
        q = html.escape(str(mq["question"]))
        reason = html.escape(str(mq.get("reason", "")))
        source_quote = str(mq.get("source_quote", "")).strip()

        sub_items = "".join(
            f'<div class="sub-q-item">{html.escape(str(s))}</div>'
            for s in mq.get("sub_questions", [])
        )

        reason_html = (
            f'<div class="reason-box"><span class="label">질문이유</span>{reason}</div>'
            if reason else ""
        )

        source_html = ""
        if source_quote:
            quote_escaped = html.escape(source_quote)
            q_no = find_source_question_no(source_quote, qa_list)
            if q_no:
                ref_text = f"출처: 자기소개서 문항 {q_no}"
            else:
                ref_text = "출처: 자기소개서 (정확한 문항 매칭 안됨)"
            source_html = (
                f'<div class="source-box">'
                f'<span class="label">근거구절</span>'
                f'<span class="quote">“{quote_escaped}”</span>'
                f'<span class="source-ref">{ref_text}</span>'
                f'</div>'
            )

        parts.append(f"""
        <div class="main-q-block">
          <div class="main-q-header">
            <div><span class="main-q-no">{html.escape(get_main_q_label(i))}</span></div>
            <div class="main-q-text">{q}</div>
            {source_html}
            {reason_html}
          </div>
          <div class="sub-q-list">
            <div class="sub-title">보조 질문</div>
            {sub_items if sub_items else '<div class="sub-q-item">(보조질문 없음)</div>'}
          </div>
        </div>""")
    return "\n".join(parts) if parts else "<p>생성된 면접 질문이 없습니다.</p>"


def render_report_body(exam_no, field, intro_qa, main_questions):
    """리포트 본문(헤더+섹션들)만 렌더링 - 개별 파일과 통합 페이지에서 공통 사용"""

    logo_uri = get_logo_data_uri()
    logo_html = (
        f'<img src="{logo_uri}" alt="한국마사회 로고" class="logo-img">'
        if logo_uri else ""
    )

    return f"""
  <div class="header">
    <div class="header-top">
      {logo_html}
      <h1>2026년 한국마사회 전임직·위촉직채용 개별면접질문</h1>
    </div>
    <div class="meta">
      <span class="badge">면접번호: {html.escape(str(exam_no))}</span>
      <span class="badge">모집분야: {html.escape(str(field or ""))}</span>
      <span class="badge">이름: ●●●(블라인드)</span>
    </div>
  </div>

  <div class="section">
    <h2>1. 맞춤형 면접 질문 (정신자세 · 전문지식 · 발표력 · 품행 · 발전가능성 - 각 보조질문 3개, 총 15개)</h2>
    {render_questions_html(main_questions, intro_qa)}
  </div>

  <div class="section">
    <h2>2. 자기소개서 원본</h2>
    {render_intro_html(intro_qa)}
  </div>

  <div class="footer">
    본 리포트는 생성형 AI 기술을 활용하여 지원자가 작성한 자기소개서 내용을 바탕으로 맞춤 제작되었습니다.
  </div>
"""


# ----------------------------------------------------------------------
# 개별 리포트 HTML
# ----------------------------------------------------------------------
INDIVIDUAL_TEMPLATE = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>2026년 한국마사회 위촉직채용 개별면접질문 - {exam_no}</title>
<style>
  body {{ background: #f4f6f8; margin: 0; padding: 24px; }}
{common_style}
</style>
</head>
<body>
<div class="container">
{body}
</div>
</body>
</html>
"""


def build_individual_report(exam_no, field, intro_qa, main_questions):
    body = render_report_body(exam_no, field, intro_qa, main_questions)
    return INDIVIDUAL_TEMPLATE.format(
        exam_no=html.escape(str(exam_no)),
        common_style=COMMON_STYLE,
        body=body,
    )


# ----------------------------------------------------------------------
# 통합 리포트(사이드바) HTML
# ----------------------------------------------------------------------
COMBINED_TEMPLATE = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>2026년 한국마사회 위촉직채용 개별면접질문 - 통합 보기</title>
<style>
  * {{ box-sizing: border-box; }}
  html, body {{
    margin: 0;
    height: 100%;
    font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif;
    background: #f4f6f8;
  }}
  .layout {{
    display: flex;
    height: 100vh;
  }}
  .sidebar {{
    width: 260px;
    flex-shrink: 0;
    background: #0f2f57;
    color: #fff;
    overflow-y: auto;
    padding: 16px 0;
  }}
  .sidebar h2 {{
    font-size: 14px;
    color: #9fc1e8;
    padding: 0 16px;
    margin: 0 0 10px 0;
    text-transform: uppercase;
    letter-spacing: 1px;
  }}
  .sidebar .item {{
    display: block;
    width: 100%;
    text-align: left;
    background: none;
    border: none;
    color: #d6e4f7;
    padding: 10px 16px;
    font-size: 14px;
    cursor: pointer;
    border-left: 3px solid transparent;
  }}
  .sidebar .item .field {{
    display: block;
    font-size: 11px;
    color: #8fb2dd;
    margin-top: 2px;
  }}
  .sidebar .item:hover {{
    background: #16407a;
  }}
  .sidebar .item.active {{
    background: #144782;
    border-left-color: #00A597;
    color: #fff;
    font-weight: bold;
  }}
  .content {{
    flex: 1;
    overflow-y: auto;
    padding: 24px;
  }}
  .report-panel {{
    display: none;
    max-width: 980px;
    margin: 0 auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    overflow: hidden;
  }}
  .report-panel.active {{
    display: block;
  }}
  .placeholder {{
    max-width: 980px;
    margin: 60px auto;
    text-align: center;
    color: #999;
    font-size: 15px;
  }}
{common_style}
</style>
</head>
<body>
<div class="layout">
  <div class="sidebar">
    <h2>지원자 목록 ({count}명)</h2>
    {sidebar_items}
  </div>
  <div class="content">
    <div class="placeholder" id="placeholder">좌측 목록에서 면접번호를 선택하세요.</div>
    {panels}
  </div>
</div>
<script>
  function showReport(examNo) {{
    document.querySelectorAll('.report-panel').forEach(function(p) {{
      p.classList.toggle('active', p.id === 'panel-' + examNo);
    }});
    document.querySelectorAll('.sidebar .item').forEach(function(b) {{
      b.classList.toggle('active', b.dataset.examNo === examNo);
    }});
    var ph = document.getElementById('placeholder');
    if (ph) ph.style.display = 'none';
  }}

  // 첫 번째 지원자를 기본으로 표시
  document.addEventListener('DOMContentLoaded', function() {{
    var first = document.querySelector('.sidebar .item');
    if (first) showReport(first.dataset.examNo);
  }});
</script>
</body>
</html>
"""


def build_combined_report(all_reports):
    """all_reports: list of (exam_no, field, intro_qa, main_questions)"""
    sidebar_items = []
    panels = []

    for exam_no, field, intro_qa, main_questions in all_reports:
        exam_no_str = html.escape(str(exam_no))
        field_str = html.escape(str(field or ""))

        sidebar_items.append(f"""
        <button class="item" data-exam-no="{exam_no_str}" onclick="showReport('{exam_no_str}')">
          {exam_no_str}
          <span class="field">{field_str}</span>
        </button>""")

        body = render_report_body(exam_no, field, intro_qa, main_questions)
        panels.append(f"""
        <div class="report-panel" id="panel-{exam_no_str}">
        {body}
        </div>""")

    return COMBINED_TEMPLATE.format(
        common_style=COMMON_STYLE,
        count=len(all_reports),
        sidebar_items="\n".join(sidebar_items),
        panels="\n".join(panels),
    )


# ----------------------------------------------------------------------
# main
# ----------------------------------------------------------------------
def main():
    global LOGO_PATH

    # 아래 세 경로를 본인 환경에 맞게 수정하세요.
    intro_path = r"자기소개서"
    interview_path = r"면접질문 생성결과"
    output_dir = r"리포트 저장경로"

    os.makedirs(output_dir, exist_ok=True)

    # output_dir 안에 LOGO_FILENAME(kra_logo.png) 파일이 있으면 자동으로 로고를 삽입합니다.
    # 다른 위치의 로고를 쓰려면 아래 줄에 절대경로를 직접 지정하세요.
    LOGO_PATH = os.path.join(output_dir, LOGO_FILENAME)

    intro_data = load_intro(intro_path)
    interview_data = load_interview_questions(interview_path)

    all_exam_nos = sorted(set(intro_data.keys()) | set(interview_data.keys()), key=str)

    print(f"총 {len(all_exam_nos)}명 처리")

    all_reports = []  # for combined report
    for exam_no in all_exam_nos:
        intro = intro_data.get(exam_no, {"모집분야": "", "qa": []})
        interview = interview_data.get(exam_no, {"모집분야": intro.get("모집분야", ""), "main_questions": []})

        field = intro.get("모집분야") or interview.get("모집분야") or ""
        intro_qa = intro["qa"]
        main_questions = interview["main_questions"]

        # 1) 개별 리포트
        report_html = build_individual_report(exam_no, field, intro_qa, main_questions)
        out_path = os.path.join(output_dir, f"면접리포트_{exam_no}.html")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(report_html)
        print(f"  - {out_path}")

        all_reports.append((exam_no, field, intro_qa, main_questions))

    # 2) 통합 리포트(사이드바)
    combined_html = build_combined_report(all_reports)
    combined_path = os.path.join(output_dir, "전체리포트.html")
    with open(combined_path, "w", encoding="utf-8") as f:
        f.write(combined_html)
    print(f"  - {combined_path}  (좌측 탭 선택 통합 리포트)")

    print("완료")


if __name__ == "__main__":
    main()

총 48명 처리
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-01.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-02.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-03.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-04.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-05.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-06.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-07.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-08.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-09.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-10.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_A-11.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_B-01.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_B-02.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_C-01.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_C-02.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포트_C-03.html
  - C:\Users\LG\Desktop\마사회 개별 리포트\리포트(최신)\면접리포

In [15]:
from bs4 import BeautifulSoup
from pathlib import Path
import re

html_path = Path(input("통합 HTML 경로 입력 : ").strip())

with open(html_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

# CSS 추출
style_tag = soup.find("style")
style_text = str(style_tag) if style_tag else ""

# report-panel 수집
panels = soup.select("div.report-panel")

field_groups = {}

for panel in panels:

    meta = panel.select_one(".meta")

    if not meta:
        continue

    badges = meta.select(".badge")

    exam_no = ""
    recruit_field = ""

    for badge in badges:

        txt = badge.get_text(" ", strip=True)

        if txt.startswith("면접번호:"):
            exam_no = txt.replace("면접번호:", "").strip()

        elif txt.startswith("모집분야:"):
            recruit_field = txt.replace("모집분야:", "").strip()

    if recruit_field:

        field_groups.setdefault(
            recruit_field,
            []
        ).append(
            (exam_no, panel)
        )

output_dir = html_path.parent / "모집분야별"
output_dir.mkdir(exist_ok=True)

for field_name, reports in field_groups.items():

    sidebar_html = f"""
<div class="sidebar">
<h2>지원자 목록 ({len(reports)}명)</h2>
"""

    for exam_no, _ in reports:

        sidebar_html += f"""
<button class="item"
        data-exam-no="{exam_no}"
        onclick="showReport('{exam_no}')">
    {exam_no}
</button>
"""

    sidebar_html += "</div>"

    content_html = """
<div class="content">
<div class="placeholder" id="placeholder">
좌측 목록에서 면접번호를 선택하세요.
</div>
"""

    for _, panel in reports:
        content_html += str(panel)

    content_html += "</div>"

    page_html = f"""
<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="utf-8">
<title>{field_name}</title>

{style_text}

<script>

function showReport(examNo) {{

    document
        .querySelectorAll(".report-panel")
        .forEach(el => el.classList.remove("active"));

    document
        .querySelectorAll(".item")
        .forEach(el => el.classList.remove("active"));

    let panel =
        document.getElementById(
            "panel-" + examNo
        );

    if(panel)
        panel.classList.add("active");

    let btn =
        document.querySelector(
            '[data-exam-no="' +
            examNo +
            '"]'
        );

    if(btn)
        btn.classList.add("active");

    let placeholder =
        document.getElementById(
            "placeholder"
        );

    if(placeholder)
        placeholder.style.display = "none";
}}

</script>

</head>

<body>

<div class="layout">

{sidebar_html}

{content_html}

</div>

</body>
</html>
"""

    safe_name = re.sub(
        r'[\\/:*?"<>|]',
        "_",
        field_name
    )

    output_file = (
        output_dir /
        f"{safe_name}.html"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(page_html)

    print(f"생성 완료 : {safe_name}.html")

print("\n모든 모집분야별 HTML 생성 완료")
print(f"저장위치 : {output_dir}")

통합 HTML 경로 입력 :  C:\Users\LG\Desktop\마사회 개별 리포트\리포트(개인별)\전체리포트.html


생성 완료 : 경마보안(과천).html
생성 완료 : 전문인력양성지원.html
생성 완료 : 말산업교육지원.html
생성 완료 : 도핑검사법운용.html
생성 완료 : 출발보조.html
생성 완료 : 출발보조(과천).html
생성 완료 : 방송기술지원.html
생성 완료 : 출발보조(부경).html
생성 완료 : 응급구조(부경).html
생성 완료 : 말관리(제주).html
생성 완료 : 경마보안(제주).html
생성 완료 : 불법경마단속.html

모든 모집분야별 HTML 생성 완료
저장위치 : C:\Users\LG\Desktop\마사회 개별 리포트\리포트(개인별)\모집분야별
